# Credit Card Default Prediction

This project builds a classification model to predict whether a customer
will default on their credit card payment in the following month.

The dataset contains information about 30,000 credit card customers from
Taiwan, including credit limits, demographic details, repayment history,
bill amounts and previous payments.

Source: UCI Machine Learning Repository

In [1]:
# Import useful libraries used in the entire project
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
# Mac-specific certificate setup
import ssl
import urllib.request
import certifi

ssl_context = ssl.create_default_context(cafile=certifi.where())

https_handler = urllib.request.HTTPSHandler(context=ssl_context)
opener = urllib.request.build_opener(https_handler)

urllib.request.install_opener(opener)

In [ ]:
# Importing Data
from ucimlrepo import fetch_ucirepo 

default_cc_clients = fetch_ucirepo(id=350)

In [ ]:
# Load data
X = default_cc_clients.data.features.copy()
y = default_cc_clients.data.targets.copy()

cc_dataset = pd.concat([X,y], axis=1)

# Rename columns using meaningful names
column_names = {
    "X1": "credit_limit_ntd",
    "X2": "gender",
    "X3": "education_level",
    "X4": "marital_status",
    "X5": "age",

    "X6": "repayment_status_sep",
    "X7": "repayment_status_aug",
    "X8": "repayment_status_jul",
    "X9": "repayment_status_jun",
    "X10": "repayment_status_may",
    "X11": "repayment_status_apr",

    "X12": "bill_amount_sep_ntd",
    "X13": "bill_amount_aug_ntd",
    "X14": "bill_amount_jul_ntd",
    "X15": "bill_amount_jun_ntd",
    "X16": "bill_amount_may_ntd",
    "X17": "bill_amount_apr_ntd",

    "X18": "payment_amount_sep_ntd",
    "X19": "payment_amount_aug_ntd",
    "X20": "payment_amount_jul_ntd",
    "X21": "payment_amount_jun_ntd",
    "X22": "payment_amount_may_ntd",
    "X23": "payment_amount_apr_ntd",

    "Y": "default_payment_next_month"
}

cc_dataset = cc_dataset.rename(columns=column_names)

cc_dataset.head()

In [ ]:
cc_dataset.tail()

In [ ]:
cc_dataset.sample(10, random_state=42)

In [ ]:
cc_dataset.shape

In [ ]:
schema = pd.DataFrame(
    {
        "dtype":cc_dataset.dtypes,
        "non-null":cc_dataset.notna().sum(),
        "missing":cc_dataset.isna().sum(),
        "missing_pct":cc_dataset.isna().mean()*100,
        "unique_values":cc_dataset.nunique(dropna=False),
    }
).sort_values("missing_pct", ascending=False)
schema

In [ ]:
duplicate_rows = cc_dataset.duplicated().sum()

print("Number of duplicate rows:", duplicate_rows)

In [ ]:
cc_dataset.drop(columns=['gender','education_level','marital_status']).describe().T #gender','education_level','marital_status these are considered as categorical variables and will study separately

#### Explore Target Variable

In [ ]:
target = "default_payment_next_month"
counts = cc_dataset[target].value_counts()
pct = cc_dataset[target].value_counts(normalize=True).mul(100).round(2).astype(str).add('%')

summary = (counts.astype(str) + ' ' + pct).rename(index={0:'No default', 1: 'Default in the following month'})
print(f"Class Distribution\n{summary}")

In [ ]:
ax = counts.rename(index={0: 'No default', 1: 'Default in the following month'}) \
           .plot(kind='bar', figsize=(12, 7))
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_title('Class Distribution of a Target Variable')
plt.tight_layout()
plt.show()


In [ ]:
category_columns = ["gender", "education_level", "marital_status"]

category_mappings = {
    "gender": {
        1: "Male",
        2: "Female"
    },
    "education_level": {
        1: "Graduate School",
        2: "University",
        3: "High School",
        4: "Others"
    },
    "marital_status": {
        1: "Married",
        2: "Single",
        3: "Others"
    }
}

category_tables = []

for column in category_columns:
    codes = sorted(cc_dataset[column].dropna().unique())

    table = pd.DataFrame({
        "column": column,
        "code": codes
    })

    table["meaning"] = (
        table["code"]
        .map(category_mappings[column])
        .fillna("Undocumented / Unknown")
    )

    category_tables.append(table)

category_table = pd.concat(category_tables, ignore_index=True)

category_table

### Findings from categorical variables

- `gender` contains only the documented codes 1 and 2.
- `education_level` contains documented codes 1–4, along with undocumented codes 0, 5 and 6.
- `marital_status` contains documented codes 1–3, along with undocumented code 0.
- The undocumented values will temporarily be treated as "Undocumented / Unknown". No values will be removed or replaced without further analysis.

In [ ]:
# How many customers and what percentage of customers belong to each code in gender, education_level, and marital_status?
category_tables = []

for column in category_columns:
    counts = cc_dataset[column].value_counts(dropna=False)
    pct = cc_dataset[column].value_counts(normalize=True).mul(100).round(2)

    table = pd.DataFrame({
        "column": column,
        "code": counts.index,
        "count": counts.values,
        "percentage": pct.values
    })
    table["meaning"] = (
        table["code"]
        .map(category_mappings[column])
        .fillna("Undocumented / Unknown")
    )
    category_tables.append(table)

category_summary = pd.concat(category_tables, ignore_index=True)
category_summary

# From the above following question can be answered
- Which category has the highest percentage in each column?
  - Gender: Female with 60.37%
  - Education Level: University 46.77%
  - Marital Status: Single is at 53.21%
- What total percentage of education_level is undocumented (0, 5, and 6 combined)? 1.15%
- Is the undocumented percentage large enough to seriously affect the analysis? 1.15% is not large enough but we should analyse whether it may impact the output.
- Based on these results, would you keep the undocumented values as a separate category or remove those records? I would keep them until further analysis.

#### Do customers with undocumented education codes have a noticeably different default rate compared with the documented education groups?

In [ ]:
cc_dataset.groupby("education_level")["default_payment_next_month"].value_counts(normalize=True).mul(100).round(2)

In [ ]:
cc_dataset["education_level"].value_counts().sort_index()

# From the above following question can be answered
- Which documented education group has the highest default rate? Education Level documented - High School has highest default rate of 25.16%
- What are the default rates for undocumented codes 0, 5, and 6? Education level undocumented - For 0: 0.00%, for 5: 6.43% and for 6: 15.69%
- Are these undocumented groups large enough for reliable comparison? A high or low percentage does not make a group reliable; the customer count matters. For example, code 6 has 15.69% default, but this represents only about 8 defaults among 51 customers.
- Should we delete them, keep them separately, or combine them as Unknown? For now, these should not deleted. Will investigate whether codes 0, 5, and 6 should be combined into one Unknown category.

- If education codes 0, 5, and 6 are combined, what are their total customer count, total defaults, and combined default rate?

In [ ]:
# Identify customers with undocumented education codes
unknown_education = cc_dataset["education_level"].isin([0, 5, 6])

# Select their target values
unknown_group = cc_dataset.loc[
    unknown_education,
    "default_payment_next_month"
]

# Calculate the results
total_customers = unknown_group.count()
total_defaults = unknown_group.sum()
default_rate = unknown_group.mean() * 100

print("Total customers:", total_customers)
print("Total defaults:", total_defaults)
print("Combined default rate:", round(default_rate, 2), "%")

### Observation

- Education codes 0, 5 and 6 are not clearly defined in the official dataset documentation.
- Together, these codes represent 345 customers.
- Among them, 26 customers defaulted.
- Their combined default rate is 7.54%.
- These customers represent only 1.15% of the complete dataset.
- Because the individual undocumented groups are small, their separate default rates may not be reliable.
- Therefore, these codes will be retained and combined into one `Unknown` category instead of deleting the records.

## Stratified Train–Test Split

The dataset is divided into training and test sets before performing further target-based analysis.

- **Training set:** 80% of the data, used for EDA and model development.
- **Test set:** 20% of the data, kept untouched for final model evaluation.
- The target column, `default_payment_next_month`, is passed to the `stratify` parameter.
- Stratification helps both sets retain approximately the original class distribution:
  - **No default (`0`):** 77.88%
  - **Default (`1`):** 22.12%
- This makes the test set more representative of the complete dataset and provides a fairer evaluation.
- `random_state=42` is used to make the split reproducible.

After this split, analytical and preprocessing decisions will be based only on the training data to avoid information leaking from the test set.

In [ ]:
from sklearn.model_selection import train_test_split

target = "default_payment_next_month"

X = cc_dataset.drop(columns=target)
y = cc_dataset[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training set class distribution:\n")
display(
    y_train.value_counts(normalize=True)
           .mul(100)
           .round(2)
           .sort_index()
)

print("\nTest set class distribution:\n")
display(
    y_test.value_counts(normalize=True)
          .mul(100)
          .round(2)
          .sort_index()
)

In [ ]:
display(f'X_train: {X_train.shape}')
display(f'X_test: {X_test.shape}')
display(f'y_train: {y_train.shape}')
display(f'y_test: {y_test.shape}')

In [ ]:
training_data = pd.concat([X_train, y_train], axis=1)
training_data.head()

In [ ]:
training_data.shape

In [ ]:
# Identify customers with undocumented education codes in training data
unknown_education_training_data = training_data["education_level"].isin([0, 5, 6])

# Select their target values
unknown_group_training_data = training_data.loc[
    unknown_education_training_data,
    "default_payment_next_month"
]

# Calculate the results
total_customers = unknown_group_training_data.count()
total_defaults = unknown_group_training_data.sum()
default_rate = unknown_group_training_data.mean() * 100

print("Total customers:", total_customers)
print("Total defaults:", total_defaults)
print("Combined default rate:", round(default_rate, 2), "%")

In [ ]:
training_data['marital_status'].value_counts(normalize=True).mul(100).round(2)

In [ ]:
unknown_marital = training_data['marital_status'].eq(0)
unknown_marital_group = training_data.loc[
    unknown_marital, "default_payment_next_month"
]
# Calculate the results
total_customers = unknown_marital_group.count()
total_defaults = unknown_marital_group.sum()
default_rate = unknown_marital_group.mean() * 100

print("Total customers:", total_customers)
print("Total defaults:", total_defaults)
print("Combined default rate:", round(default_rate, 2), "%")

### Findings: Undocumented Education Codes in Training Data

- Education codes `0`, `5`, and `6` are not defined in the official dataset documentation.
- Together, these codes represent **289 customers** in the training data.
- Among them, **22 customers defaulted**.
- Their combined default rate is **7.61%**, compared with the overall training default rate of approximately **22.12%**.
- Since the individual undocumented groups are small, their separate default rates may not be reliable.
- These customers will be retained, while codes `0`, `5`, and `6` will be provisionally combined into one `Unknown` category during preprocessing.

### Findings: Undocumented Marital-Status Code in Training Data

- Marital-status code `0` is not defined in the official dataset documentation.
- It represents **47 customers**, or approximately **0.20%** of the training data.
- Among them, **5 customers defaulted**.
- Their default rate is **10.64%**.
- Since this group is very small, its default rate is not reliable enough for a strong conclusion.
- These customers will be retained, while marital-status code `0` will be provisionally labelled as `Unknown` during preprocessing.

### Provisional Preprocessing Decision

- `education_level` codes `0`, `5`, and `6` → `Unknown`
- `marital_status` code `0` → `Unknown`
- No customer records will be deleted.
- The original training data will remain unchanged; these transformations will be applied during preprocessing.

In [ ]:
training_data.columns

In [ ]:
repayment_columns = [
    column for column in training_data.columns if column.startswith('repayment_status')
]

In [ ]:
repayment_columns

In [ ]:
%matplotlib inline 
import matplotlib.pyplot as plt
training_data[repayment_columns].hist(bins=12, figsize=(20,15))
plt.show()

In [ ]:
for column in repayment_columns:
    print(column, sorted(training_data[column].dropna().unique()))

In [ ]:
training_data[repayment_columns[0]].value_counts()

In [ ]:
training_data[repayment_columns[0]].value_counts(normalize=True).mul(100).round(2)

### Observation: September Repayment Status

- Code `0` is the most frequent repayment status, representing 11,781 customers or 49.09% of the training data.
- Codes `-1` and `-2` represent 18.86% and 9.16%, respectively.
- Although codes `0` and `-2` occur frequently in the dataset, their meanings are not clearly defined in the official documentation.
- Their high frequencies suggest that they are genuine operational categories rather than isolated data-entry errors.
- Therefore, codes `-2`, `-1`, and `0` will be retained as separate categories without assigning undocumented meanings.

In [ ]:
total_customers = training_data.groupby(repayment_columns[0])["default_payment_next_month"].count()
total_default = training_data.groupby(repayment_columns[0])["default_payment_next_month"].sum()
total_default_rate = training_data.groupby(repayment_columns[0])["default_payment_next_month"].mean()

# Print results
print(f"Total Customers: {total_customers}")
print(f"Total Default: {total_default}")
print(f"Total Default Rate: {total_default_rate}")

### Interpretation

- Repayment-status code `2` provides stronger evidence of high default risk.
- It has a default rate of 69.36%, based on 1,483 defaults among 2,138 customers.
- Code `8` has a 50% default rate, but it contains only 16 customers and 8 defaults.
- Therefore, the rate for code `8` is statistically unstable and should be interpreted cautiously.
- High percentages based on very small groups should not be compared directly with rates based on large groups.

In [ ]:
results = training_data.groupby(column)["default_payment_next_month"].agg(
    total_customers="count",
    total_defaults="sum",
    default_rate="mean"
)
results

In [ ]:
print("Customers in grouped results:", results["total_customers"].sum())
print("Customers in training data:", len(training_data))

print("\nDefaults in grouped results:", results["total_defaults"].sum())
print("Customers in training data:", y_train.sum())